# Day 1 — Hit all five FortyGuard endpoints by hand

**Owner: T2** · Save every raw response to `data/fixtures/`. Log credit balance somewhere visible.

Endpoints (submit → poll `/v1/status/{activity_id}`):

1. `heatmap`
2. `satellite`
3. `streetview`
4. `heat_intelligence`
5. `env_params`

Plus: `GET /v1/system/fetch-api-key-usage` for the credit meter.

> Put your key in `.env` (never commit it). Run with network on. After success, commit the fixtures and switch to `REPLAY_MODE=true`.


In [17]:
from __future__ import annotations

import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import httpx
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.environ.get("FORTYGUARD_API_KEY")
if not API_KEY or API_KEY.startswith("your_"):
    raise SystemExit("Set FORTYGUARD_API_KEY in .env (copy from .env.example)")

BASE = "https://api.fortyguard.com/v1"
HEADERS = {"api-key": API_KEY, "Content-Type": "application/json"}

# Relative to repo root when run from notebooks/
FIXTURES = Path("../data/fixtures")
FIXTURES.mkdir(parents=True, exist_ok=True)

STAMP = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
print(f"Fixture stamp: {STAMP}")
print(f"Writing to: {FIXTURES.resolve()}")


Fixture stamp: 20260822_085533
Writing to: C:\Users\Kailey Lam\Desktop\Hackathons\workface\data\fixtures


In [18]:
def save(name: str, payload: dict | list) -> Path:
    path = FIXTURES / f"{name}_{STAMP}.json"
    path.write_text(json.dumps(payload, indent=2, default=str))
    print(f"  saved → {path.name}")
    return path

def submit(endpoint: str, body: dict) -> str:
    url = f"{BASE}/{endpoint}"
    print(f"\nPOST {url}")
    r = httpx.post(url, headers=HEADERS, json=body, timeout=60.0)
    print(f"  status={r.status_code}")
    data = r.json()
    save(f"{endpoint}_submit", data)
    if r.status_code >= 400:
        raise RuntimeError(f"Submit failed: {data}")
    aid = data["data"]["activity_id"]
    print(f"  activity_id={aid}")
    return aid

def poll(activity_id: str, endpoint: str, max_wait_s: int = 180, interval_s: float = 3.0) -> dict:
    url = f"{BASE}/status/{activity_id}"
    deadline = time.time() + max_wait_s
    attempt = 0
    while time.time() < deadline:
        attempt += 1
        r = httpx.get(url, headers=HEADERS, timeout=30.0)
        data = r.json()
        status = (data.get("data") or {}).get("status") or data.get("message")
        print(f"  poll #{attempt}: {status}")
        if status in ("Completed", "Failed"):
            save(f"{endpoint}_result", data)
            return data
        time.sleep(interval_s * (1.2 ** min(attempt, 5)))  # mild backoff
    raise TimeoutError(f"Timed out waiting for {activity_id}")


## 0. Credit balance (call first and last)

In [3]:
def fetch_credits() -> dict:
    # Tech spec: /v1/system/fetch-api-key-usage
    url = f"{BASE}/system/fetch-api-key-usage"
    print(f"GET {url}")
    r = httpx.get(url, headers=HEADERS, timeout=30.0)
    print(f"  status={r.status_code}")
    data = r.json()
    save("credit_balance", data)
    # Surface visibly
    print("\n========== CREDIT BALANCE ==========")
    print(json.dumps(data, indent=2, default=str))
    print("====================================\n")
    return data

credits_before = fetch_credits()


GET https://api.fortyguard.com/v1/system/fetch-api-key-usage
  status=405
  saved → credit_balance_20260822_071626.json

========== CREDIT BALANCE ==========
{
  "detail": "Method Not Allowed"
}



## Demo geometry — North Phoenix fab campus (August)

Pinned AOI for the sprint: semiconductor fab campus, North Phoenix.
All five Day-1 captures use this polygon / point and an August timestamp.


In [19]:
# North Phoenix semiconductor fab campus (approx TSMC Arizona area).
# GeoJSON is [longitude, latitude]. Closed ring.
POLYGON_AOI = {
    "type": "FeatureCollection",
    "features": [
        {
            "type": "Feature",
            "properties": {"name": "north_phoenix_fab_campus"},
            "geometry": {
                "type": "Polygon",
                "coordinates": [[
                    [-112.165223, 33.784010],
                    [-112.152500, 33.784130],
                    [-112.152334, 33.771711],
                    [-112.165055, 33.771592],
                    [-112.165223, 33.784010],
                ]],
            },
        }
    ],
}

# Hero point ≈ campus centroid
LAT, LON = 33.777861, -112.158778

# August historical day inside supported window (2019-01-01 … now+12h)
DATE = "2024-08-15"
TIME = "14:00"


## 1. heatmap

`filter_type: 1` = single hour. Granularity 100 m for the cheapest first probe.


In [20]:
heatmap_body = {
    "polygon_aoi": POLYGON_AOI,
    "date_time": {
        "start_date": DATE,
        "start_time": TIME,
        "filter_type": 1,
    },
    "granularity": 100,
}

heatmap_id = submit("heatmap", heatmap_body)
heatmap_result = poll(heatmap_id, "heatmap")
print("heatmap status:", (heatmap_result.get("data") or {}).get("status"))



POST https://api.fortyguard.com/v1/heatmap
  status=200
  saved → heatmap_submit_20260822_085533.json
  activity_id=219d19a4-868a-45d7-8e1f-cdf9461e9123
  poll #1: Processing
  poll #2: Processing
  poll #3: Processing
  poll #4: Completed
  saved → heatmap_result_20260822_085533.json
heatmap status: Completed


## 2. satellite


In [21]:
satellite_body = {
    "sat": {
        "latitude": LAT,
        "longitude": LON,
    },
    "date_time": {
        "start_date": DATE,
        "start_time": TIME,
        "filter_type": 1,
    },
    "granularity": 80,
}

satellite_id = submit("satellite", satellite_body)
satellite_result = poll(satellite_id, "satellite")
print("satellite status:", (satellite_result.get("data") or {}).get("status"))



POST https://api.fortyguard.com/v1/satellite
  status=200
  saved → satellite_submit_20260822_085533.json
  activity_id=e3acc5ae-cb08-49c0-b288-360c65b52956
  poll #1: Processing
  poll #2: Completed
  saved → satellite_result_20260822_085533.json
satellite status: Completed


## 3. streetview


In [23]:
streetview_body = {
    "latitude": LAT,
    "longitude": LON,
    "vertical_angle": 10.0,
    "horizontal_angle": 90.0,
    "back_view": False,
}

streetview_id = submit("streetview", streetview_body)
streetview_result = poll(streetview_id, "streetview")
print("streetview status:", (streetview_result.get("data") or {}).get("status"))



POST https://api.fortyguard.com/v1/streetview
  status=200
  saved → streetview_submit_20260822_085533.json
  activity_id=7f16c622-5570-4dd2-bbb2-c94e62dc5fe5
  poll #1: Failed
  saved → streetview_result_20260822_085533.json
streetview status: Failed


In [13]:
streetview_id = "72f815b6-3dbe-496f-a66a-bfc579708e75"

streetview_result = poll(
    streetview_id,
    "streetview",
    max_wait_s=300,   # 10 min
    interval_s=5.0
)

status = (streetview_result.get("data") or {}).get("status")
print("status:", status)

# Always keep whatever came back
save("streetview_result_final", streetview_result)

  poll #1: Processing
  poll #2: Processing
  poll #3: Processing
  poll #4: Processing
  poll #5: Processing
  poll #6: Processing
  poll #7: Processing
  poll #8: Processing
  poll #9: Processing
  poll #10: Processing
  poll #11: Processing
  poll #12: Processing
  poll #13: Processing
  poll #14: Processing
  poll #15: Processing
  poll #16: Processing
  poll #17: Processing
  poll #18: Processing
  poll #19: Processing
  poll #20: Processing
  poll #21: Processing
  poll #22: Processing
  poll #23: Processing


TimeoutError: Timed out waiting for 72f815b6-3dbe-496f-a66a-bfc579708e75

## 4. heat_intelligence

Requires a temperature that matches a heatmap you generated for the same location/time.
Use a plausible value (e.g. 40.0 °C for Phoenix August) for the first capture; later chain from real heatmap stats.


In [15]:
heat_intel_body = {
    "latitude": LAT,
    "longitude": LON,
    "temperature": 40.0,
    "date": DATE,
    "analysis": ["environmental", "geographic", "urban", "events", "anthropogenic"],
}

heat_intel_id = submit("heat_intelligence", heat_intel_body)
heat_intel_result = poll(heat_intel_id, "heat_intelligence", max_wait_s=300)

# Redact signed download_link if present (docs: temporary; do not log full URL)
result_data = (heat_intel_result.get("data") or {}).get("result") or {}
if isinstance(result_data, dict) and "download_link" in result_data:
    link = result_data["download_link"]
    print("download_link present (redacting before final save)…")
    # Optional: fetch PDF immediately and store under a short name
    try:
        pdf_r = httpx.get(link, timeout=60.0, follow_redirects=True)
        if pdf_r.status_code == 200 and pdf_r.headers.get("content-type", "").startswith("application/pdf"):
            pdf_path = FIXTURES / f"heat_intelligence_report_{STAMP}.pdf"
            pdf_path.write_bytes(pdf_r.content)
            print(f"  PDF saved → {pdf_path.name}")
        result_data = {**result_data, "download_link": "[REDACTED — fetched and stored as PDF]"}
        heat_intel_result["data"]["result"] = result_data
        save("heat_intelligence_result_redacted", heat_intel_result)
    except Exception as e:
        print(f"  PDF fetch failed (non-fatal): {e}")

print("heat_intelligence status:", (heat_intel_result.get("data") or {}).get("status"))



POST https://api.fortyguard.com/v1/heat_intelligence
  status=200
  saved → heat_intelligence_submit_20260822_071626.json
  activity_id=e28385da-9437-4918-bfbd-07acfa51e533
  poll #1: Processing
  poll #2: Processing
  poll #3: Processing
  poll #4: Processing
  poll #5: Processing
  poll #6: Processing
  poll #7: Processing
  poll #8: Processing
  poll #9: Processing
  poll #10: Processing
  poll #11: Processing
  poll #12: Processing
  poll #13: Processing
  poll #14: Processing
  poll #15: Processing
  poll #16: Processing
  poll #17: Processing
  poll #18: Processing
  poll #19: Processing
  poll #20: Processing
  poll #21: Processing
  poll #22: Processing
  poll #23: Processing
  poll #24: Processing
  poll #25: Processing
  poll #26: Processing
  poll #27: Processing
  poll #28: Processing
  poll #29: Processing
  poll #30: Processing
  poll #31: Processing
  poll #32: Processing
  poll #33: Processing
  poll #34: Processing


TimeoutError: Timed out waiting for e28385da-9437-4918-bfbd-07acfa51e533

In [16]:
heat_intel_body = {
    "latitude": LAT,
    "longitude": LON,
    "temperature": 40.0,
    "date": DATE,
    "analysis": ["environmental", "geographic", "urban", "events", "anthropogenic"],
}

heat_intel_id = "e28385da-9437-4918-bfbd-07acfa51e533"
heat_intel_result = poll(heat_intel_id, "heat_intelligence", max_wait_s=300)

# Redact signed download_link if present (docs: temporary; do not log full URL)
result_data = (heat_intel_result.get("data") or {}).get("result") or {}
if isinstance(result_data, dict) and "download_link" in result_data:
    link = result_data["download_link"]
    print("download_link present (redacting before final save)…")
    # Optional: fetch PDF immediately and store under a short name
    try:
        pdf_r = httpx.get(link, timeout=60.0, follow_redirects=True)
        if pdf_r.status_code == 200 and pdf_r.headers.get("content-type", "").startswith("application/pdf"):
            pdf_path = FIXTURES / f"heat_intelligence_report_{STAMP}.pdf"
            pdf_path.write_bytes(pdf_r.content)
            print(f"  PDF saved → {pdf_path.name}")
        result_data = {**result_data, "download_link": "[REDACTED — fetched and stored as PDF]"}
        heat_intel_result["data"]["result"] = result_data
        save("heat_intelligence_result_redacted", heat_intel_result)
    except Exception as e:
        print(f"  PDF fetch failed (non-fatal): {e}")

print("heat_intelligence status:", (heat_intel_result.get("data") or {}).get("status"))


  poll #1: Processing
  poll #2: Processing
  poll #3: Processing
  poll #4: Processing
  poll #5: Processing
  poll #6: Processing
  poll #7: Processing
  poll #8: Processing
  poll #9: Completed
  saved → heat_intelligence_result_20260822_071626.json
download_link present (redacting before final save)…
  PDF saved → heat_intelligence_report_20260822_071626.pdf
  saved → heat_intelligence_result_redacted_20260822_071626.json
heat_intelligence status: Completed


## 5. env_params

Same temperature / date_time discipline as heat_intelligence.


In [22]:
env_params_body = {
    "latitude": LAT,
    "longitude": LON,
    "temperature": 40.0,
    "date_time": {
        "start_date": DATE,
        "start_time": TIME,
        "filter_type": 1,
    },
}

env_params_id = submit("env_params", env_params_body)
env_params_result = poll(env_params_id, "env_params")
print("env_params status:", (env_params_result.get("data") or {}).get("status"))



POST https://api.fortyguard.com/v1/env_params
  status=200
  saved → env_params_submit_20260822_085533.json
  activity_id=f15e41fb-6adc-4166-ad8d-e9156c4da793
  poll #1: Completed
  saved → env_params_result_20260822_085533.json
env_params status: Completed


## Credit balance after batch (log somewhere visible)


In [ ]:
credits_after = fetch_credits()

# Also write a human-visible summary at repo root so it shows up in PRs / Slack
summary_path = Path("../CREDITS.md")
summary = f"""# FortyGuard credit balance

Captured: {datetime.now(timezone.utc).isoformat()}

## Before batch
```json
{json.dumps(credits_before, indent=2, default=str)}
```

## After batch
```json
{json.dumps(credits_after, indent=2, default=str)}
```

Fixtures written under `data/fixtures/` with stamp `{STAMP}`.
"""
summary_path.write_text(summary)
print(f"Wrote visible log → {summary_path.resolve()}")


## Done checklist

- [ ] All five endpoints returned Completed (or you recorded Failed + activity_id)
- [ ] Raw JSON under `data/fixtures/`
- [ ] `CREDITS.md` shows balance before/after
- [ ] `.env.example` committed (no real secrets)
- [ ] Fixtures committed — they become the REPLAY demo

Next: implement `apps/api/fortyguard/client.py` against these fixtures (`REPLAY_MODE=true` by default).
